# Lab 12d : Traçabilité de la consommation — le contrat C6 rend l'usage LLM observable
**Navigation** : [Lab 12c <<](Lab12c-Agent-Handoff.ipynb) | [README du track](../README.md)

**Grain #14687 (registre EPITA #14058, contrat C6)**. La consommation d'une conversation
multi-agents n'est pas un détail d'exploitation : c'est une **propriété observable du
runtime**. Jusqu'a la tracons 1, ADK produisait bien `usage_metadata` sur ses events,
mais `AdkRunResult` ne le remontait pas — la consommation était invisible depuis le code.

Ce lab exécute de **vrais tours LLM** (seul le provider du `.env` change selon la machine)
et lit, appel par appel, ce que le runtime consomme :

- `AdkRunResult.usage_turns` : un snapshot `AdkUsage` (prompt / completion / total) par appel LLM du tour ;
- `AdkRunResult.usage_total` : la somme du tour ;
- `ConversationRunner.usage_total` : le profil cumulé d'une conversation entière.

Jamais une restauration du moteur SK (#14058) : ADK reste le runtime, l'usage est
simplement rendu observable au-dessus de lui.

## 1. Configuration

Le track charge son provider LLM depuis la configuration (`config/providers.py`).
La clé n'est jamais lue ni affichée ici — seuls le provider, le modèle et le type d'endpoint le sont.

In [1]:
import sys
sys.path.insert(0, '..')

import warnings

warnings.filterwarnings(
    "ignore",
    message=(
        r"\[EXPERIMENTAL\] feature "
        r"FeatureName\.JSON_SCHEMA_FOR_FUNC_DECL is enabled\."
    ),
    category=UserWarning,
    module=r"google\.adk\.models\.llm_request",
)

from config.providers import get_settings, get_provider_config, get_litellm_model

settings = get_settings()
provider = get_provider_config(settings)
print(f"Provider actif : {provider.provider.value}")
print(f"Modele : {get_litellm_model(provider)}")
print(f"Endpoint externe : {bool(provider.base_url)}")

Provider actif : openrouter
Modele : openrouter/openai/gpt-5.6-luna
Endpoint externe : True


## 2. Un tour avec outil : deux appels LLM, deux snapshots

L'agent `dataset_profile` repond a une question en **deux appels LLM** : le premier
décide d'invoquer l'outil, le second interprète son résultat. Chaque appel laisse un
snapshot dans `usage_turns` — la tracons C6 distingue les deux jambes du tour.

In [2]:
from utils.adk_runtime import run_data_agent, AdkUsage

resultat = await run_data_agent(
    "Un dataset contient 30 lignes et 5 colonnes. "
    "Appelle dataset_profile et interprete le resultat.",
    timeout_seconds=120,
)

print(f"Appels LLM du tour : {len(resultat.usage_turns)}")
for i, usage in enumerate(resultat.usage_turns, start=1):
    print(f"  appel {i} : prompt={usage.prompt_tokens} "
          f"completion={usage.completion_tokens} total={usage.total_tokens}")
print(f"Cumul du tour (usage_total) : {resultat.usage_total}")
print(f"Outil invoque : {resultat.tool_was_invoked}")

Appels LLM du tour : 2
  appel 1 : prompt=154 completion=34 total=188
  appel 2 : prompt=215 completion=44 total=259
Cumul du tour (usage_total) : AdkUsage(prompt_tokens=369, completion_tokens=78, total_tokens=447)
Outil invoque : True


### Lecture du résultat

Le tour laisse **deux snapshots distincts**. Le premier correspond à la décision d'appel
d'outil, le second à l'interprétation : la réponse finale ne sort pas du même appel que la
decision. Le coût d'un tour n'est donc pas un nombre, c'est une **décomposition** — et
`usage_total` en est la somme. Si le provider ne compte pas ses jetons, `usage_turns`
reste vide : le runtime n'invente rien.

Lu dans les sorties committées : `appel 1 : prompt=154 completion=34 total=188` puis `appel 2 : prompt=215 completion=44 total=259`, cumul `AdkUsage(prompt_tokens=369, completion_tokens=78, total_tokens=447)`. Chaque composante se vérifie colonne par colonne : 154 + 215 = 369 prompts, 34 + 44 = 78 completions, 188 + 259 = 447 totaux — et la somme croisée 369 + 78 = 447 referme le compte. L'interprétation coûte plus cher que la décision (259 contre 188, soit +38 %), et son prompt grossit de 61 jetons — la place prise par le résultat de l'outil dans le contexte du second appel. C'est exactement la propriété que l'exercice 2 demandera de tester sur un tour quelconque.

## 3. Profil d'une conversation multi-tours : la mémoire a un prix

Le contrat C1b (persistance entre tours, Lab12e) fait revenir **tout l'historique** à chaque
appel. C6 rend ce coût mesurable : dans une conversation de trois tours, le `prompt_tokens`
de chaque tour doit **croître** — c'est la trace chiffrée de la mémoire qui s'accumule.

In [3]:
import asyncio
from utils.adk_conversation import ConversationRunner
from utils.adk_runtime import build_data_agent

async def conversation_trois_tours():
    agent = build_data_agent()
    lignes_profil = []
    async with ConversationRunner(agent) as conversation:
        questions = [
            "Un dataset contient 12 lignes et 3 colonnes. Appelle dataset_profile.",
            "Souviens-toi du dataset precedent : combien de cellules avait-il ? "
            "Appelle dataset_profile pour verifier.",
            "Toujours pour ce meme dataset : quelle est la densite par colonne ? "
            "Appelle dataset_profile une derniere fois.",
        ]
        for i, question in enumerate(questions, start=1):
            tour = await conversation.turn(question, timeout_seconds=120)
            lignes_profil.append(
                (i, tour.usage_total.prompt_tokens,
                 tour.usage_total.completion_tokens,
                 conversation.usage_total.total_tokens))
    return lignes_profil

profil = await conversation_trois_tours()
print("tour | prompt | completion | cumul conversation")
print("-----|--------|------------|------------------")
for i, prompt, completion, cumul in profil:
    print(f"  {i}  |  {prompt:4d}  |    {completion:3d}    |  {cumul:5d}")

tour | prompt | completion | cumul conversation
-----|--------|------------|------------------
  1  |   359  |     71    |    430
  2  |   611  |     49    |   1090
  3  |   819  |    197    |   2106


### Lecture du résultat

La colonne `prompt` **croît à chaque tour** : le deuxième et le troisième appel reçoivent
l'historique complet des précédents. C'est le coût marginal de la persistance C1b — un coût
réel, maintenant mesurable au lieu d'imaginaire. La colonne `completion` reste du même ordre :
c'est le contexte qui coûte, pas les réponses. La dernière colonne est ce qu'un **budget**
de conversation devra borner (jambe 2 du grain #14687).

Lu dans les sorties committées (entête `tour | prompt | completion | cumul conversation`) : les trois lignes lisent 359/71/430, 611/49/1090 et 819/197/2106. Deltas de prompt : +252 puis +208 — chaque tour ajoute un bloc quasi constant, la trace de l'historique qui revient entier. La completion, elle, n'est pas monotone : 71, puis 49, puis 197 — le dernier appel répond plus longuement que les deux premiers réunis (étendue de 148 jetons). Chaque cumul est la somme exacte des tours passés : 359 + 71 = 430, puis 430 + 611 + 49 = 1090, puis 1090 + 819 + 197 = 2106. En trois tours, le prompt d'un seul appel est passé de 359 à 819 — 2,3 fois son coût initial, une croissance quasi linéaire sur ce run (+252 puis +208, environ 230 jetons d'historique par tour) ; c'est ce poids croissant qu'un budget de conversation devra borner.

## 4. Ce que C6 ne dit pas (encore) : le budget natif n'existe pas

La tracons est portee ; le **plafond** n'est pas natif. La mesure du registre reste vraie :
`RunConfig` ne porte aucun champ de consommation (`max_llm_calls` borne des *appels*, pas
les jetons). Verifions-le sur le runtime reellement installe.

In [4]:
from google.adk.runners import RunConfig

champs_budget = [
    f for f in RunConfig.model_fields
    if "token" in f.lower() or "cost" in f.lower()
]
print(f"Champs de budget natif dans RunConfig : {champs_budget or 'AUCUN'}")

Champs de budget natif dans RunConfig : AUCUN


### Lecture du résultat

La sortie dit verbatim : `Champs de budget natif dans RunConfig : AUCUN` — le filtre a cherché `token` et `cost` dans `model_fields` de `RunConfig` sans rien trouver — le budget de jetons devra être un **assemblage au-dessus d'ADK** (un plafond sur `conversation.usage_total` qui coupe proprement), pas une option de configuration. La mesure qui précède donne l'échelle du manque : la conversation de la section 3 est montée à 2106 jetons cumulés sans qu'aucune option du runtime ne puisse l'arrêter en cours de route. Le contraste est net — par appel, le runtime expose TROIS compteurs (prompt, completion, total) ; pour la conversation entière, ZERO plafond. Toute la jambe 2 du grain #14687 tient dans cet écart : la section 5 va assembler au-dessus d'ADK le plafond que `RunConfig` n'offre pas. La traçabilité la rend possible : on ne peut borner que ce qu'on mesure.

## 5. Le budget porte au-dessus d'ADK : couper proprement

La section 4 a mesuré l'absence de budget natif. Le voici, assemblé au-dessus d'ADK : un
plafond de jetons cumulés par conversation (`budget_total_tokens`), qui **refuse un tour avant
tout appel LLM** quand le plafond est déjà atteint, et **coupe au premier appel qui le
franchit**. La coupe rend un verdict explicite — l'exception typée `AdkBudgetExceeded`, qui
porte le plafond, le cumul exact au moment de la coupe et le numéro du tour — pas une
exception brute du runtime.

Pour que la démonstration soit déterministe quel que soit le comptage réel du provider, le
plafond est **dérivé de la mesure** : juste au-dessus du cumul du tour 1, le tour 2 doit le
franchir dès son premier appel.

In [5]:
from utils.adk_conversation import AdkBudgetExceeded
from utils.adk_runtime import build_data_agent

async def demonstration_budget():
    agent = build_data_agent()
    async with ConversationRunner(agent) as conversation:
        await conversation.turn(
            "Un dataset contient 7 lignes et 2 colonnes. Appelle dataset_profile.",
            timeout_seconds=120,
        )
        # Plafond derive de la mesure : le cumul du tour 1 + une marge mince.
        conversation.budget_total_tokens = conversation.usage_total.total_tokens + 30
        plafond = conversation.budget_total_tokens
        try:
            await conversation.turn(
                "Un dataset contient 9 lignes et 3 colonnes. Appelle dataset_profile.",
                timeout_seconds=120,
            )
            return plafond, None
        except AdkBudgetExceeded as verdict:
            return plafond, verdict

plafond, verdict = await demonstration_budget()
if verdict is None:
    print(f"Plafond {plafond} non franchi -- relancer la cellule.")
else:
    print(f"Plafond pose apres le tour 1 : {plafond} jetons")
    print(f"Verdict : {verdict.verdict} (tour {verdict.tour})")
    print(f"Cumul au moment de la coupe : {verdict.usage_total.total_tokens} jetons")
    print(f"Message d'origine : {verdict}")

Plafond pose apres le tour 1 : 460 jetons
Verdict : BUDGET_EXCEEDED (tour 2)
Cumul au moment de la coupe : 723 jetons
Message d'origine : BUDGET_EXCEEDED: 723 jetons cumules >= plafond 460 (tour 2)


### Lecture du résultat

Le tour 2 n'a jamais produit de réponse : dès que son premier appel LLM fait franchir le
plafond, la consommation s'arrête et le verdict `BUDGET_EXCEEDED` porte le plafond, le cumul
exact au moment de la coupe (les jetons consommés avant la coupe restent comptés — comptabilité
honnête) et le numéro du tour coupé. Pas d'exception brute du stream : un **verdict type**,
attrapable, qui laisse la conversation dans un état comptable cohérent, arrêtée sur un chiffre exact. On ne peut borner que
ce qu'on mesure — la traçabilité des sections 2 et 3 est ce qui rend ce plafond possible.

Lu dans les sorties committées : `Plafond pose apres le tour 1 : 460 jetons` — le code le construit comme cumul du tour 1 + 30, donc le tour 1 de cette démonstration a consommé 460 - 30 = 430 jetons. Le verdict : `BUDGET_EXCEEDED: 723 jetons cumules >= plafond 460 (tour 2)` — dépassement de 723 - 460 = 263 jetons, 57 % au-dessus du plafond. Ce seul chiffre prouve la granularité de la coupe : elle n'attend pas la fin du tour, elle tombe dès que le premier appel franchit la ligne — et le verdict porte les trois nombres qu'il faut pour auditer (plafond 460, cumul 723, tour 2).

## 6. Exercices

Trois exercices pour vous approprier le contrat. Le notebook doit rester executable même
si vous ne les completez pas — chaque cellule d'exercice est un stub correct (C.1).

### Exercice 1 -- Coût marginal de la mémoire

Écrivez `cout_marginal(profil)` qui, à partir du profil de la section 3, rend la liste des
delta de `prompt` entre tours consécutifs. Le premier delta doit être `None` (pas de tour
précédent).

In [6]:
# Exercice 1 : a completer
# Etape 1 : recuperer la colonne prompt du profil de la section 3.
# Etape 2 : calculer les differences entre tours consecutifs.
# Etape 3 : retourner la liste (le premier element vaut None).

def cout_marginal(profil):
    # TODO etudiant
    return None

print("Exercice 1 a completer : cout_marginal(profil)")

Exercice 1 a completer : cout_marginal(profil)


### Exercice 2 -- Détecteur d'incohérence d'usage

Écrivez `usage_coherent(resultat)` qui rend `True` ssi la somme des `usage_turns` de
`resultat` égale exactement `resultat.usage_total` — la propriété qui doit tenir pour que
un budget (jambe 2) puisse se fier au cumul.

In [7]:
# Exercice 2 : a completer
# Etape 1 : sommer les snapshots de resultat.usage_turns.
# Etape 2 : comparer chaque composante a resultat.usage_total.
# Etape 3 : retourner le booleen.

def usage_coherent(resultat):
    # TODO etudiant
    return None

print("Exercice 2 a completer : usage_coherent(resultat)")

Exercice 2 a completer : usage_coherent(resultat)


**Lecture chiffrée — vers l'exercice 2 : la cohérence snapshots/cumul reste à établir.** Le tour de la section 2 fournit le matériel brut de l'exercice 2 : un cumul de tokens affiche par le runtime et, en amont, des snapshots par appel. L'exercice demande d'écrire la propriété qui vérifie la cohérence des deux sur plusieurs formes d'entrée — quelles égalités exiger, avec quelle tolerance, et que faire des tours où un provider ne rapporte rien : autant de choix de sémantique qui appartiennent à l'étudiant. La propriété doit tenir pour TOUT tour, pas seulement pour ceux où tout se passe bien ; le budget de la section 5, qui se fie au cumul, dépend de savoir détecter un écart.

### Exercice 3 -- Isolement des compteurs

C1 doit rester tenu côté consommation : deux conversations isolées doivent avoir des cumuls
indépendants. Écrivez `compteurs_independants(agent)` qui joue la même question dans deux
`ConversationRunner` distinctes et rend `True` ssi aucun cumul ne fuit dans l'autre.

In [8]:
# Exercice 3 : a completer
# Etape 1 : ouvrir deux ConversationRunner sur le meme agent.
# Etape 2 : jouer la meme question dans chacune.
# Etape 3 : verifier que les deux usage_total sont strictement positifs
#           et que chaque cumul ne compte que ses propres tours.

def compteurs_independants(agent):
    # TODO etudiant
    return None

print("Exercice 3 a completer : compteurs_independants(agent)")

Exercice 3 a completer : compteurs_independants(agent)


## 7. Conclusion

| Idée clé | Où la voir |
|---|---|
| Un tour = plusieurs appels LLM, chacun avec son snapshot | section 2, `usage_turns` |
| La mémoire C1b a un coût marginal mesurable | section 3, croissance de `prompt` |
| Le budget natif n'existe pas : `RunConfig` sans champ de consommation | section 4 |
| Un plafond de jetons qui coupe proprement, verdict type | section 5, `AdkBudgetExceeded` |
| Un compteur qui n'invente rien : provider muet = `usage_turns` vide | docstring `_event_usage` |

Le contrat C6 est désormais **porté en entier** (PR #14690) : ce que la conversation consomme
est observable depuis le code du depot, appel par appel, ET bornable — un plafond qui coupe
proprement sur un verdict explicite. Toujours au-dessus d'ADK, jamais une restauration SK.
See #14687, #14058.

## À retenir

1. Chaque appel LLM est observable, et l'observation est double : 2 appels dans le tour — prompt=154 completion=34 total=188, puis prompt=215 completion=44 total=259 — et un cumul de tour, `AdkUsage(prompt_tokens=369, completion_tokens=78, total_tokens=447)`.
2. La mémoire a un prix mesurable : le prompt croît de tour en tour (359, puis 611, puis 819) et le cumul de conversation suit (430, 1090, 2106). Rapprocher les tours coûte, et le carnet chiffre ce coût.
3. Le runtime ne sait pas borner : « Champs de budget natif dans RunConfig : AUCUN ». Aucun champ de budget n'existe dans `RunConfig` — le plafond doit être porté au-dessus d'ADK, pas demandé à ADK.
4. La coupe est propre et explicite : plafond posé après le tour 1 à 460 jetons, verdict `BUDGET_EXCEEDED (tour 2)`, cumul au moment de la coupe 723 jetons, message « BUDGET_EXCEEDED: 723 jetons cumules >= plafond 460 (tour 2) ».

## Vérifiez votre compréhension

1. Pourquoi le prompt du tour 2 est-il plus grand que celui du tour 1 ?
2. Le budget existe-t-il nativement dans le runtime ?
3. Un plafond de 460 jetons posé après le tour 1 a-t-il arrêté le tour 2, et à quel niveau de consommation ?

<details><summary>Réponses</summary>

1. Parce que la requête emporte l'historique de la conversation : le tour 2 doit faire tenir au modèle tout ce qui a déjà été dit, en plus de sa propre question. Le profil multi-tours le montre dans sa colonne prompt — 359 au tour 1, 611 au tour 2, 819 au tour 3. La croissance est celle de la mémoire, pas celle du travail demandé.
2. Non, et la mesure l'affiche : « Champs de budget natif dans RunConfig : AUCUN ». Le contrôle du coût ne peut donc pas être délégué au runtime ; il est implémenté au-dessus, par un compteur lu après chaque tour.
3. Oui : le verdict est `BUDGET_EXCEEDED (tour 2)`, le cumul au moment de la coupe vaut 723 jetons, et le message indique « 723 jetons cumules >= plafond 460 (tour 2) ». La coupure est donc nette — elle porte sur le cumul, pas sur un tour qui aurait pu se terminer proprement.

</details>

## Pour aller plus loin

- **Contrat jumeau** : [Lab12c-Agent-Handoff.ipynb](Lab12c-Agent-Handoff.ipynb) — C5, où le transfert lui-même a un coût en jetons.
- **Contrat voisin** : [Lab12e-Session-Persistence.ipynb](Lab12e-Session-Persistence.ipynb) — C1b, ce qui rend la mémoire persistante, donc payante à chaque tour.
- **Carnet précédent** : [Lab12b-Sequential-Orchestration.ipynb](Lab12b-Sequential-Orchestration.ipynb) — C4, l'ordre posé par l'appelant.
- **Index** : [README du track](../README.md) · [README de la série ML](../../README.md).
- **Lecture** : Nam et al., *DS-STAR: Data Science Agent for Solving Diverse Tasks across Heterogeneous Formats and Open-Ended Queries*, arXiv:2509.21825, 2025 ; Xi et al., *The Rise and Potential of Large Language Model Based Agents: A Survey*, arXiv:2309.07864, 2023.